In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Delta-based Gain Provenance Audit

Two distinct analyses are reported separately:
  1) Two-stage  : molecule_type has a genuine raw -> normalized -> canonical
                  chain, so both syntax-stabilization and canonical-field
                  effects are measurable.
  2) One-stage  : species / disease / sample_name undergo lexical
                  stabilization only (no separate raw column exists), so we
                  honestly report exact -> stabilized retrieval only and do
                  NOT fabricate a canonical-field delta for these fields.

"""

import pandas as pd
from pathlib import Path

from evisionary_common import (
    DATA_PATH_MASTER, MOLECULE_CONCEPTS, QUERY_PATTERNS, valid_text_series,
    contains_ci, count_unique_pmids, summarize_sources, safe_fold_change,
)
OUT_DIR = Path("/Users/sogand/Desktop/Article_Figures/keyB/Validation_Audits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH_MASTER)
print(f"Loaded {len(df):,} rows from key_B master table.")

# ---------------------------------------------------------
# TWO-STAGE: molecule_type (genuine raw -> norm -> canonical chain)
# ---------------------------------------------------------
two_stage_rows = []

raw_col = "molecule_type_raw"
canon_col = "molecule_type"  # canonical exported field

for concept, exact_val, pattern in MOLECULE_CONCEPTS:
    raw_s = valid_text_series(df[raw_col])
    canon_s = valid_text_series(df[canon_col])

    m_exact = raw_s.str.casefold().eq(exact_val)
    m_stable = contains_ci(raw_s, pattern)
    m_harm = contains_ci(canon_s, pattern)

    c_exact, c_stable, c_harm = int(m_exact.sum()), int(m_stable.sum()), int(m_harm.sum())

    # Defensive: skip any concept entirely absent from the data.
    if c_harm == 0 and c_exact == 0:
        print(f"[skip] '{concept}' absent in molecule_type; omitted from table.")
        continue

    two_stage_rows.append({
        "Query_Concept": concept,
        "Analysis_Type": "Two-stage (raw->norm->canonical)",
        "Raw_Field": raw_col,
        "Canonical_Field": canon_col,
        "Raw_Exact_Records": c_exact,
        "Raw_Query_Stable_Records": c_stable,
        "Canonical_Harmonized_Records": c_harm,
        "Delta_Syntax_Stabilization": c_stable - c_exact,
        "Delta_Canonical_Field": c_harm - c_stable,
        "Gain_over_Raw_Exact": c_harm - c_exact,
        "Fold_Change_vs_Raw_Exact": safe_fold_change(c_exact, c_harm),
        "Unique_PMIDs_Harmonized": count_unique_pmids(df, m_harm),
        "Source_Distribution_Harmonized": summarize_sources(df, m_harm),
    })

# ---------------------------------------------------------
# ONE-STAGE: species / disease / sample_name
# No separate raw column -> honestly report exact -> stabilized only.
# ---------------------------------------------------------
one_stage_rows = []

stabilization_concepts = [
    ("Homo sapiens",      "species",     "homo sapiens",      QUERY_PATTERNS["Homo sapiens"]),  # ← اینجا
    ("Rattus norvegicus", "species",     "rattus norvegicus", r"\brattus\s+norvegicus\b"),
    ("Plasma",            "sample_name", "plasma",            r"\bplasma\b"),
    ("Breast Cancer",     "disease",     "breast cancer",     r"\bbreast[-\s]+cancer\b"),
]

for concept, field, exact_val, pattern in stabilization_concepts:
    s = valid_text_series(df[field])
    m_exact = s.str.casefold().eq(exact_val)
    m_stable = contains_ci(s, pattern)
    c_exact, c_stable = int(m_exact.sum()), int(m_stable.sum())

    one_stage_rows.append({
        "Query_Concept": concept,
        "Analysis_Type": "One-stage (lexical stabilization only)",
        "Field": field,
        "Exact_Match_Records": c_exact,
        "Stabilized_Records": c_stable,
        "Stabilization_Gain": c_stable - c_exact,
        "Fold_Change": safe_fold_change(c_exact, c_stable),
        "Unique_PMIDs_Stabilized": count_unique_pmids(df, m_stable),
        "Source_Distribution_Stabilized": summarize_sources(df, m_stable),
        "Note": "No separate raw column exists for this field; "
                "only syntax stabilization is measurable.",
    })

# ---------------------------------------------------------
# Export
# ---------------------------------------------------------
pd.DataFrame(two_stage_rows).to_csv(
    OUT_DIR / "Table_Gain_TwoStage_MoleculeType.csv", index=False)
pd.DataFrame(one_stage_rows).to_csv(
    OUT_DIR / "Table_Gain_OneStage_Stabilization.csv", index=False)

print("\n=== Two-stage (molecule_type) ===")
print(pd.DataFrame(two_stage_rows).to_string(index=False))
print("\n=== One-stage (stabilization only) ===")
print(pd.DataFrame(one_stage_rows).to_string(index=False))
print(f"\nOutputs written to: {OUT_DIR}")

Loaded 258,460 rows from key_B master table.

=== Two-stage (molecule_type) ===
Query_Concept                    Analysis_Type         Raw_Field Canonical_Field  Raw_Exact_Records  Raw_Query_Stable_Records  Canonical_Harmonized_Records  Delta_Syntax_Stabilization  Delta_Canonical_Field  Gain_over_Raw_Exact  Fold_Change_vs_Raw_Exact  Unique_PMIDs_Harmonized       Source_Distribution_Harmonized
      Protein Two-stage (raw->norm->canonical) molecule_type_raw   molecule_type             207623                    207623                        207623                           0                      0                    0                       1.0                      569 Vesiclepedia:160806 | ExoCarta:46817
         mRNA Two-stage (raw->norm->canonical) molecule_type_raw   molecule_type              26701                     26701                         26701                           0                      0                    0                       1.0                       26   Vesicle